In [1]:
from overseer import *

Spinnaker python SDK installation not found
Spinnaker python SDK installation not found
ximea python SDK installation not found
PI python SDK installation not found


In [2]:
bigbrother = Overseer()

In [8]:
bigbrother.set_last_offset_to_current()

In [26]:
offset_new = bigbrother.get_tcs_total_offset()
bigbrother.update_guidebox_from_offset(offset_new)

In [5]:
PLATE_SCALE = 0.16 # "/pixel
POS_ANGLE = 2.3    # degrees, align image to sky coordinates

In [15]:
import subprocess

def get_tcs_total_offset():
    """Retrieves the current guide box offsets from t3io.
    """
    result = subprocess.run(["t3io", "info", "OS"], capture_output=True, text=True)
    output = result.stdout.strip()
    output = output.split(' ')
    if output[0] != "OK":
        raise ConnectionError(f"t3io error. Output: {output}")
    
    # Format is OK TotalOS(ra dec) UserOS(ra dec enable) BeamOS(ra dec enable) ScanOS(ra dec) [all in arcsec]
    # We only care about the total offset
    offset_new = ( float(output[1]), float(output[2]) )
    return offset_new

#offset_last = get_tcs_total_offset()
offset_new = get_tcs_total_offset()

In [16]:
def update_guidebox_from_offset(offset, offset_last):
    """Changes the subaperture masks (basically the guidebox) based differences
    between the last offset and current offset reported by t3io.
    """
    dra = offset[0] - offset_last[0]
    ddec = offset[1] - offset_last[1]

    # Convert to pixel offsets
    dx = dra / PLATE_SCALE
    dy = ddec / PLATE_SCALE
    print("dx, dy are", dx, dy)

    # rotate ccw by the position angle to align with the camera coordinates
    theta = np.radians(POS_ANGLE)
    dx_rot = dx * np.cos(theta) - dy * np.sin(theta)
    dy_rot = dx * np.sin(theta) + dy * np.cos(theta)
    dx = dx_rot
    dy = dy_rot

    print("post rotation:", dx, dy)

    # subap masks are defined in terms of the final binned image
    binning = 1
    dx /= binning
    dy /= binning
    dx = round(dx)
    dy = round(dy)
    
    # Update the subApMasks.
    x0, y0 = 0, 0
    cx = int(x0 + dx)
    cy = int(y0 + dy)

    print(cx, cy)

update_guidebox_from_offset(offset_new, offset_last)

dx, dy are 6.25 0.0
post rotation: 6.244964969465287 0.25082370332849835
6 0
